# Сравнение методов разметки волновода

6 методов автоматической сегментации waveguide. Запусти и сравни — скажи какой ближе к правде.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly
from pathlib import Path
import os

PROJECT = Path(os.path.abspath(''))
if not (PROJECT / 'config.py').exists():
    PROJECT = Path(os.path.expanduser('~/Desktop/НИР/ДаняБоряНир'))
DATA = PROJECT.parent / 'data' / 'dataset'
RESULTS = PROJECT / 'results'

# 4 тестовых кадра (по одному из каждого видео, без аугментации)
TEST_FRAMES = {
    'MVI_6265': 'MVI_6265_MOV-0000_jpg.rf.88deb6daaf538a9c5a4da89486b6448e',
    'MVI_6268': 'MVI_6268_MOV-0000_jpg.rf.752f74a512d87c320e009f7da06b3b90',
    'MVI_6270': 'MVI_6270_MOV-0000_jpg.rf.af488334580ccc06c58107d262bd3217',
    'MVI_6273': 'MVI_6273_MOV-0002_jpg.rf.289c1a662be63a152730fd0bb3d265c0',
}

def load_frame(vid_key):
    """Загружает кадр и его label."""
    name = TEST_FRAMES[vid_key]
    img = cv2.imread(str(DATA / 'images' / 'train' / (name + '.jpg')))
    if img is None:
        # Попробуем val/test
        for split in ['val', 'test']:
            img = cv2.imread(str(DATA / 'images' / split / (name + '.jpg')))
            if img is not None:
                break
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None
    return img, img_rgb

def load_original_annotation(vid_key):
    """Загружает оригинальную Roboflow аннотацию waveguide."""
    name = TEST_FRAMES[vid_key]
    for split in ['train', 'val', 'test']:
        lbl_path = DATA / 'labels' / split / (name + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if int(parts[0]) == 0:  # waveguide
                        coords = list(map(float, parts[1:]))
                        pts = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
                        return pts
    return None

def mask_to_contour(mask):
    """Конвертирует бинарную маску в контур (list of (x,y) pixel coords)."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    return biggest.squeeze().tolist()

def draw_result(ax, img_rgb, contour_px, title, color='red'):
    """Рисует результат метода на оси."""
    ax.imshow(img_rgb)
    if contour_px is not None and len(contour_px) >= 3:
        poly = MplPoly(contour_px, closed=True, facecolor=color, edgecolor=color,
                       alpha=0.3, linewidth=2)
        ax.add_patch(poly)
        xs = [p[0] for p in contour_px]
        ys = [p[1] for p in contour_px]
        # Зум на waveguide
        margin = 80
        ax.set_xlim(min(xs)-margin, max(xs)+margin)
        ax.set_ylim(max(ys)+margin, min(ys)-margin)
        area = cv2.contourArea(np.array(contour_px, dtype=np.int32))
        title += f'\n{len(contour_px)} pts, {area:.0f} px²'
    else:
        title += '\nНЕ НАЙДЕН'
    ax.set_title(title, fontsize=9)
    ax.axis('off')

# Проверка загрузки
for vid in TEST_FRAMES:
    img, _ = load_frame(vid)
    print(f'{vid}: {"OK" if img is not None else "FAIL"} ({img.shape if img is not None else "N/A"})')

---
## Метод 1: Тёмная полость → расширение на стенки
Ищем самый тёмный прямоугольный блоб (полость волновода), затем расширяем на толщину стенок.

In [ ]:
def method_dark_cavity(img, dark_thresh=55, wall_expand=18, bottom_extra=25):
    """Метод 1: Найти тёмную полость → расширить на стенки."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Маска тёмных пикселей
    dark = (gray < dark_thresh).astype(np.uint8) * 255
    
    # Фокус на центральной области
    mask_center = np.zeros_like(dark)
    mask_center[h//5:4*h//5, w//6:5*w//6] = 255
    dark = cv2.bitwise_and(dark, mask_center)
    
    # Морфология
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k)
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    
    # Найти контуры полости
    contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    for c in contours:
        a = cv2.contourArea(c)
        if a < 300:
            continue
        x, y, cw, ch = cv2.boundingRect(c)
        aspect = cw / max(ch, 1)
        if 0.3 < aspect < 3.0 and 20 < cw < 250 and 20 < ch < 250:
            if best is None or a > cv2.contourArea(best):
                best = c
    
    if best is None:
        return None, dark
    
    cx, cy, ccw, cch = cv2.boundingRect(best)
    
    # Расширяем на стенки
    x1 = max(0, cx - wall_expand)
    y1 = max(0, cy - wall_expand)
    x2 = min(w, cx + ccw + wall_expand)
    y2 = min(h, cy + cch + bottom_extra)
    
    # Создаём контур из прямоугольника
    contour = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
    return contour, dark

# Тест
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, dark_mask = method_dark_cavity(img)
    
    # Верхний ряд: результат
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 1: Dark Cavity', 'red')
    
    # Нижний ряд: промежуточная маска
    axes[1, col].imshow(dark_mask, cmap='gray')
    axes[1, col].set_title('Маска тёмных пикселей', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Метод 1: Тёмная полость → расширение', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 2: GrabCut
Инициализируем GrabCut прямоугольником вокруг центральной области. Алгоритм сам разделит передний/задний план.

In [ ]:
def method_grabcut(img, iterations=8):
    """Метод 2: GrabCut с прямоугольной инициализацией."""
    h, w = img.shape[:2]
    
    # Сначала найдём примерную область через яркость
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Яркая область = стенки волновода
    bright = (gray > 170).astype(np.uint8) * 255
    center_mask = np.zeros_like(bright)
    center_mask[h//4:3*h//4, w//5:4*w//5] = 255
    bright = cv2.bitwise_and(bright, center_mask)
    bright = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
    
    contours, _ = cv2.findContours(bright, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, np.zeros((h, w), dtype=np.uint8)
    
    biggest = max(contours, key=cv2.contourArea)
    bx, by, bw, bh = cv2.boundingRect(biggest)
    
    # Слегка расширим bbox для GrabCut
    margin = 15
    rect = (max(0, bx-margin), max(0, by-margin), 
            min(bw+2*margin, w-bx+margin), min(bh+2*margin, h-by+margin))
    
    gc_mask = np.zeros((h, w), np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    
    cv2.grabCut(img, gc_mask, rect, bgd, fgd, iterations, cv2.GC_INIT_WITH_RECT)
    
    fg = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    
    # Фокус на центральной части
    fg = cv2.bitwise_and(fg, center_mask)
    
    # Морфология: убрать шум, сгладить
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (10, 10)))
    
    contour = mask_to_contour(fg)
    return contour, fg

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, gc_mask = method_grabcut(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 2: GrabCut', 'lime')
    axes[1, col].imshow(gc_mask, cmap='gray')
    axes[1, col].set_title('GrabCut маска', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Метод 2: GrabCut', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 3: K-means цветовая кластеризация
Кластеризуем пиксели по цвету, находим кластер, соответствующий стенкам волновода (самый яркий в центре).

In [ ]:
def method_kmeans(img, K=6):
    """Метод 3: K-means кластеризация цвета."""
    h, w = img.shape[:2]
    
    # Кластеризация
    pixels = img.reshape((-1, 3)).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
    labels = labels.reshape(h, w)
    
    # Центральная область
    cy_start, cy_end = h//4, 3*h//4
    cx_start, cx_end = w//5, 4*w//5
    
    # Ищем: 1) самый тёмный кластер в центре (полость)
    #        2) самый яркий кластер в центре (стенки)
    center_labels = labels[cy_start:cy_end, cx_start:cx_end]
    
    # Яркость кластеров
    brightness = [c.mean() for c in centers]
    
    # Считаем присутствие каждого кластера в центре
    center_counts = {}
    for k in range(K):
        center_counts[k] = np.sum(center_labels == k)
    
    # Самый яркий кластер, который присутствует в центре (стенки)
    # Фильтруем: кластер должен занимать хотя бы 2% центральной области
    min_center_px = 0.02 * center_labels.size
    wall_candidates = [(k, brightness[k]) for k in range(K) 
                       if center_counts[k] > min_center_px and brightness[k] > 150]
    wall_candidates.sort(key=lambda x: x[1], reverse=True)
    
    # Самый тёмный (полость)
    dark_candidates = [(k, brightness[k]) for k in range(K)
                       if center_counts[k] > min_center_px * 0.5 and brightness[k] < 80]
    dark_candidates.sort(key=lambda x: x[1])
    
    # Строим маску: стенки + полость
    mask = np.zeros((h, w), dtype=np.uint8)
    if wall_candidates:
        mask |= ((labels == wall_candidates[0][0]) * 255).astype(np.uint8)
    if dark_candidates:
        mask |= ((labels == dark_candidates[0][0]) * 255).astype(np.uint8)
    
    # Фокус на центре
    center_fmask = np.zeros_like(mask)
    center_fmask[h//5:4*h//5, w//6:5*w//6] = 255
    mask = cv2.bitwise_and(mask, center_fmask)
    
    # Морфология
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    
    contour = mask_to_contour(mask)
    
    # Визуализация кластеров
    segmented = centers[labels.flatten()].reshape(img.shape).astype(np.uint8)
    return contour, mask, cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, kmask, seg_vis = method_kmeans(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 3: K-means', 'cyan')
    axes[1, col].imshow(kmask, cmap='gray')
    axes[1, col].set_title('Маска (стенки+полость)', fontsize=9)
    axes[1, col].axis('off')
    axes[2, col].imshow(seg_vis)
    axes[2, col].set_title(f'K-means кластеры (K=6)', fontsize=9)
    axes[2, col].axis('off')

plt.suptitle('Метод 3: K-means цветовая кластеризация', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 4: Canny → поиск прямоугольного контура
Детекция краёв → аппроксимация контуров прямоугольниками → фильтрация по размеру и форме.

In [ ]:
def method_canny_rect(img, canny_low=50, canny_high=150):
    """Метод 4: Canny edges → поиск прямоугольных контуров."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Блюр для уменьшения шума
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # CLAHE для улучшения контраста
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)
    
    # Canny
    edges = cv2.Canny(enhanced, canny_low, canny_high)
    
    # Замыкание краёв
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    
    # Фокус на центре
    center_mask = np.zeros_like(edges)
    center_mask[h//5:4*h//5, w//6:5*w//6] = 255
    edges_center = cv2.bitwise_and(edges, center_mask)
    
    # Найти контуры
    contours, _ = cv2.findContours(edges_center, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    # Ищем прямоугольные контуры подходящего размера
    candidates = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < 1000 or area > 80000:
            continue
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.04 * peri, True)
        x, y, cw, ch = cv2.boundingRect(c)
        extent = area / (cw * ch) if cw * ch > 0 else 0
        # Прямоугольность: 4 вершины и высокий extent
        candidates.append((c, area, len(approx), extent, x, y, cw, ch))
    
    # Сортируем: предпочитаем прямоугольные (4 вершины), средний размер
    # Целевой размер: ~5000-20000 px² (waveguide + стенки)
    target_area = 10000
    candidates.sort(key=lambda c: (abs(c[1] - target_area) / target_area + 
                                    abs(c[2] - 4) * 0.3 +
                                    (1 - c[3]) * 0.5))
    
    if candidates:
        best = candidates[0]
        contour = best[0].squeeze().tolist()
        return contour, edges_center
    return None, edges_center

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, edges = method_canny_rect(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 4: Canny+Rect', 'yellow')
    axes[1, col].imshow(edges, cmap='gray')
    axes[1, col].set_title('Canny edges (центр)', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Метод 4: Canny → прямоугольный контур', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 5: Watershed
Distance transform + watershed для разделения волновода от фона.

In [ ]:
def method_watershed(img):
    """Метод 5: Watershed сегментация."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Otsu бинаризация
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Фокус на центре
    center_mask = np.zeros_like(thresh)
    center_mask[h//5:4*h//5, w//6:5*w//6] = 255
    thresh = cv2.bitwise_and(thresh, center_mask)
    
    # Морфология: убрать шум
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Sure background (расширение)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Sure foreground (distance transform)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    
    # Unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Markers
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Watershed
    markers = cv2.watershed(img, markers)
    
    # Найти сегмент, ближайший к центру кадра
    unique_labels = np.unique(markers)
    best_label = None
    best_dist = float('inf')
    img_center = np.array([w/2, h * 0.55])  # чуть ниже центра
    
    for label in unique_labels:
        if label <= 1:  # background or border
            continue
        segment_mask = (markers == label).astype(np.uint8) * 255
        area = np.sum(segment_mask > 0)
        if area < 500 or area > 60000:
            continue
        # Центр масс сегмента
        ys, xs = np.where(segment_mask > 0)
        cx, cy = np.mean(xs), np.mean(ys)
        d = np.sqrt((cx - img_center[0])**2 + (cy - img_center[1])**2)
        if d < best_dist:
            best_dist = d
            best_label = label
    
    if best_label is not None:
        result_mask = (markers == best_label).astype(np.uint8) * 255
        result_mask = cv2.morphologyEx(result_mask, cv2.MORPH_CLOSE, 
                                        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
        contour = mask_to_contour(result_mask)
        return contour, (markers > 1).astype(np.uint8) * 255
    return None, (markers > 1).astype(np.uint8) * 255

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, ws_mask = method_watershed(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 5: Watershed', 'magenta')
    axes[1, col].imshow(ws_mask, cmap='gray')
    axes[1, col].set_title('Watershed сегменты', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Метод 5: Watershed', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 6: CLAHE + Otsu + морфология
Улучшаем контраст через CLAHE, бинаризуем Otsu, морфологическая очистка.

In [ ]:
def method_clahe_otsu(img):
    """Метод 6: CLAHE + Otsu + морфология."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
    enhanced = clahe.apply(gray)
    
    # Otsu на enhanced
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Фокус на центре
    center_mask = np.zeros_like(binary)
    center_mask[h//5:4*h//5, w//6:5*w//6] = 255
    binary = cv2.bitwise_and(binary, center_mask)
    
    # Морфология: close → open
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, 
                               cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10)))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, 
                               cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    
    # Найти контур, ближайший к центру
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_center = np.array([w/2, h * 0.55])
    
    best = None
    best_dist = float('inf')
    for c in contours:
        area = cv2.contourArea(c)
        if area < 1000 or area > 60000:
            continue
        M = cv2.moments(c)
        if M['m00'] == 0:
            continue
        cx = M['m10'] / M['m00']
        cy = M['m01'] / M['m00']
        d = np.sqrt((cx - img_center[0])**2 + (cy - img_center[1])**2)
        if d < best_dist:
            best_dist = d
            best = c
    
    contour = best.squeeze().tolist() if best is not None else None
    return contour, binary, enhanced

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, binary, enhanced = method_clahe_otsu(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 6: CLAHE+Otsu', 'orange')
    axes[1, col].imshow(enhanced, cmap='gray')
    axes[1, col].set_title('CLAHE enhanced', fontsize=9)
    axes[1, col].axis('off')
    axes[2, col].imshow(binary, cmap='gray')
    axes[2, col].set_title('Otsu бинаризация', fontsize=9)
    axes[2, col].axis('off')

plt.suptitle('Метод 6: CLAHE + Otsu + морфология', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Метод 7: Яркие стенки + тёмная полость → convex hull
Комбинируем: яркие пиксели (стенки) + тёмные (полость) в центральной зоне → выпуклая оболочка.

In [ ]:
def method_bright_dark_hull(img, bright_thresh=170, dark_thresh=55):
    """Метод 7: Яркие стенки + тёмная полость → convex hull."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Центральная маска
    cmask = np.zeros((h, w), dtype=np.uint8)
    cmask[h//5:4*h//5, w//6:5*w//6] = 255
    
    # Яркие пиксели (стенки)
    bright = ((gray > bright_thresh).astype(np.uint8) * 255) & cmask
    bright = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10)))
    
    # Тёмные пиксели (полость)
    dark = ((gray < dark_thresh).astype(np.uint8) * 255) & cmask
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    
    # Объединяем
    combined = cv2.bitwise_or(bright, dark)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (12, 12)))
    
    # Контуры → convex hull самого большого в центре
    contours, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    img_center = np.array([w/2, h * 0.55])
    best = None
    best_score = float('inf')
    for c in contours:
        area = cv2.contourArea(c)
        if area < 1000 or area > 60000:
            continue
        M = cv2.moments(c)
        if M['m00'] == 0:
            continue
        cx = M['m10'] / M['m00']
        cy = M['m01'] / M['m00']
        d = np.sqrt((cx - img_center[0])**2 + (cy - img_center[1])**2)
        # Предпочитаем: ближе к центру + крупнее
        score = d - area * 0.002
        if score < best_score:
            best_score = score
            best = c
    
    if best is not None:
        hull = cv2.convexHull(best)
        contour = hull.squeeze().tolist()
        return contour, combined, bright, dark
    return None, combined, bright, dark

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for col, vid in enumerate(TEST_FRAMES):
    img, img_rgb = load_frame(vid)
    contour, combined, bright, dark = method_bright_dark_hull(img)
    draw_result(axes[0, col], img_rgb, contour, f'{vid}\nМетод 7: Bright+Dark→Hull', '#00FF88')
    # Визуализация: RGB — bright=зелёный, dark=красный
    vis3ch = np.zeros((*bright.shape, 3), dtype=np.uint8)
    vis3ch[:,:,1] = bright  # green = bright walls
    vis3ch[:,:,2] = dark    # red = dark cavity
    axes[1, col].imshow(vis3ch)
    axes[1, col].set_title('Зелёный=стенки, Красный=полость', fontsize=8)
    axes[1, col].axis('off')
    axes[2, col].imshow(combined, cmap='gray')
    axes[2, col].set_title('Объединённая маска', fontsize=9)
    axes[2, col].axis('off')

plt.suptitle('Метод 7: Яркие стенки + тёмная полость → convex hull', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Сравнение ВСЕХ методов на одном кадре
Все 7 методов рядом для каждого видео.

In [ ]:
method_names = [
    ('Оригинал\nRoboflow', 'white'),
    ('М1: Dark\nCavity', 'red'),
    ('М2:\nGrabCut', 'lime'),
    ('М3:\nK-means', 'cyan'),
    ('М4: Canny\n+Rect', 'yellow'),
    ('М5:\nWatershed', 'magenta'),
    ('М6: CLAHE\n+Otsu', 'orange'),
    ('М7: Bright\n+Dark Hull', '#00FF88'),
]

for vid in TEST_FRAMES:
    img, img_rgb = load_frame(vid)
    h, w = img.shape[:2]
    
    # Все контуры
    results = []
    
    # 0: Оригинал Roboflow
    orig = load_original_annotation(vid)
    if orig:
        results.append([(x*w, y*h) for x, y in orig])
    else:
        results.append(None)
    
    # 1: Dark cavity
    c1, _ = method_dark_cavity(img)
    results.append(c1)
    
    # 2: GrabCut
    c2, _ = method_grabcut(img)
    results.append(c2)
    
    # 3: K-means
    c3, _, _ = method_kmeans(img)
    results.append(c3)
    
    # 4: Canny+Rect
    c4, _ = method_canny_rect(img)
    results.append(c4)
    
    # 5: Watershed
    c5, _ = method_watershed(img)
    results.append(c5)
    
    # 6: CLAHE+Otsu
    c6, _, _ = method_clahe_otsu(img)
    results.append(c6)
    
    # 7: Bright+Dark Hull
    c7, _, _, _ = method_bright_dark_hull(img)
    results.append(c7)
    
    # Рисуем
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    for idx, (ax, (name, color)) in enumerate(zip(axes.flat, method_names)):
        draw_result(ax, img_rgb, results[idx], name, color)
    
    plt.suptitle(f'{vid} — сравнение всех методов', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Все методы наложены на один кадр
Все контуры разных цветов на одном изображении — видно разницу.

In [ ]:
for vid in TEST_FRAMES:
    img, img_rgb = load_frame(vid)
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img_rgb)
    
    # Оригинал
    orig = load_original_annotation(vid)
    all_xs, all_ys = [], []
    
    methods = [
        ('Roboflow',    orig and [(x*w,y*h) for x,y in orig], 'white', '--'),
        ('М1: DarkCav', method_dark_cavity(img)[0], 'red', '-'),
        ('М2: GrabCut', method_grabcut(img)[0], 'lime', '-'),
        ('М3: K-means', method_kmeans(img)[0], 'cyan', '-'),
        ('М4: Canny',   method_canny_rect(img)[0], 'yellow', '-'),
        ('М5: WaterSh', method_watershed(img)[0], 'magenta', '-'),
        ('М6: CLAHE',   method_clahe_otsu(img)[0], 'orange', '-'),
        ('М7: BrDkHull',method_bright_dark_hull(img)[0], '#00FF88', '-'),
    ]
    
    for name, contour, color, ls in methods:
        if contour and len(contour) >= 3:
            poly = MplPoly(contour, closed=True, facecolor='none', 
                          edgecolor=color, linewidth=2, linestyle=ls, label=name)
            ax.add_patch(poly)
            xs = [p[0] for p in contour]
            ys = [p[1] for p in contour]
            all_xs.extend(xs)
            all_ys.extend(ys)
    
    # Зум
    if all_xs:
        margin = 60
        ax.set_xlim(min(all_xs)-margin, max(all_xs)+margin)
        ax.set_ylim(max(all_ys)+margin, min(all_ys)-margin)
    
    ax.legend(loc='upper left', fontsize=9, framealpha=0.8)
    ax.set_title(f'{vid} — все методы наложены', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## Эталон для сравнения
Твоя ручная разметка рядом с результатами.

In [ ]:
ref_path = RESULTS / 'user_annotation_waveguide.jpg'
if ref_path.exists():
    ref_img = plt.imread(str(ref_path))
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(ref_img)
    ax.set_title('Эталон: твоя ручная разметка waveguide\n(жёлтый = стенки, фиолетовый = полость)', 
                 fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Файл эталонной разметки не найден')

print('\n' + '='*60)
print('Скажи какой метод ближе всего к правильной разметке!')
print('Можно комбинировать или тюнить параметры.')
print('='*60)